# License and Attribution

**Copyright © 2026 Randy Balzer. All rights reserved.**

These notebooks (`red_team_engagement_simulator.ipynb` and `tool_use_agentic_attacks.ipynb`) are shared publicly via GitHub for educational, research, and professional portfolio purposes.

**Attribution Requirement**  
If any portion of this work is used, adapted, extended, or incorporated into other projects, research, presentations, or commercial applications, clear and prominent attribution is required in the following form:

> Based on work by Randy Balzer (randy@balzer.io). Original notebooks: *red_team_engagement_simulator.ipynb* and *tool_use_agentic_attacks.ipynb*.

For substantial derivative works or any commercial application, please contact randy@balzer.io to discuss appropriate licensing terms.

This material is provided “as is” without warranty of any kind, express or implied. The author disclaims all liability arising from the use or misuse of these notebooks.

# Red Team Engagement Simulator

**AI Red Teaming Notebook**  
**Author:** Randy Balzer  
**Date:** June 2026  
**Focus:** Multi-agent red team engagement simulation with human-in-the-loop oversight, MITRE ATT&CK mapping, phased execution (Reconnaissance → Exploitation → Post-Exploitation → Reporting), and professional reporting

**Purpose**

This notebook implements a controlled, phased red team engagement simulator using specialized worker agents orchestrated with explicit human approval gates between phases. It demonstrates reconnaissance, exploitation simulation, post-exploitation activities, and the generation of professional engagement reports while preserving operator judgment at high-impact decision points.

**Relevance to Enterprise AI Red Teaming**

Modern security organizations are increasingly exploring agentic AI for automated red teaming, detection engineering validation, tabletop exercises, and security orchestration. A well-designed simulator allows teams to:
- Test and improve detection coverage against realistic attack chains
- Train SOC analysts on engagement workflows
- Evaluate the effectiveness of human-in-the-loop controls for autonomous agents
- Generate high-quality, MITRE-mapped artifacts for reporting and compliance

This notebook provides a lightweight yet realistic foundation that can be extended to real agent frameworks (LangGraph, CrewAI) and integrated with tool-use attack primitives demonstrated in companion notebooks.

## 1. Strategic Context & Objectives

Traditional red team engagements are manual, time-intensive, and difficult to scale or repeat consistently. Agentic systems offer the promise of automation, but they also introduce new risks: uncontrolled escalation, hallucinated techniques, or unintended actions on production-like environments.

This simulator addresses these challenges by:

1. Decomposing the engagement into discrete, auditable phases
2. Enforcing explicit human approval before progressing to higher-impact phases
3. Requiring structured MITRE ATT&CK technique mapping in exploitation and post-exploitation outputs
4. Producing a professional, actionable final report

### Notebook Objectives

- Demonstrate a practical orchestrator-worker pattern with human oversight
- Enforce high-quality MITRE ATT&CK formatting and linking
- Provide a reproducible test harness for evaluating agentic red teaming workflows
- Serve as a foundation for integrating more advanced attack primitives (tool schema poisoning, workflow hijacking) from companion notebooks

## 2. Simulator Architecture & Phased Methodology

The simulator follows a strict four-phase engagement model with mandatory human approval gates after reconnaissance and after exploitation.

| Phase                    | Agent                  | Objective                                      | Human Gate | Risk Level |
|--------------------------|------------------------|------------------------------------------------|------------|------------|
| **1. Reconnaissance**    | recon_agent            | Map attack surface, identify services & vulnerabilities | Yes        | Low        |
| **2. Exploitation**      | exploit_agent          | Simulate initial access using realistic MITRE techniques | Yes        | Medium     |
| **3. Post-Exploitation** | post_exploit_agent     | Demonstrate persistence, credential access, lateral movement | No         | High       |
| **4. Reporting**         | report_agent           | Synthesize findings into professional engagement report | No         | Low        |

**Key Design Principle**: High-impact phases (especially exploitation) require explicit human approval. This mirrors real-world rules of engagement and prevents runaway autonomous actions.

## 3. Human-in-the-Loop Design Rationale

Explicit human approval gates provide several critical benefits:

- **Risk Containment**: Prevents an agent from proceeding to exploitation or post-exploitation without operator confirmation.
- **Auditability**: Every approval decision is logged in the notebook session.
- **Judgment Augmentation**: The human operator can apply context that the LLM may lack (business impact, scope limitations, rules of engagement).
- **Training Value**: New analysts can observe the decision points that experienced red teamers consider.

Future iterations will explore replacing the simple `input()` prompts with a dedicated oversight agent that can approve low-risk actions automatically while escalating high-risk ones.

## 4. Setup, Imports, and Configuration

In [1]:
!pip install langgraph langchain-openai langchain -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 8.6 MB/s eta 0:00:00


In [2]:
# Imports and API Key
import os
from getpass import getpass
from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage

# Load OpenAI API Key securely
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

print("OpenAI API Key loaded.")

OpenAI API Key loaded.


## 5. Specialized Worker Tools

Each worker agent is intentionally narrow in scope. This follows the principle of least privilege for agents and makes behavior more predictable and auditable.

In [3]:
# Define Tools

@tool
def recon_worker(target: str) -> str:
    """Perform reconnaissance on the target and return specific findings including open ports, services, and potential vulnerabilities."""
    return (
        f"[Recon] Completed reconnaissance on {target}.\n"
        f"- Open ports: 22 (SSH), 80 (HTTP), 443 (HTTPS), 3389 (RDP)\n"
        f"- Services detected: Apache 2.4.41, OpenSSH 8.2p1, Microsoft IIS 10.0\n"
        f"- Potential vulnerabilities: Outdated Apache version, weak SSL/TLS configuration, "
        f"possible RDP exposure to the internet.\n"
        f"- Recommended focus areas: Web application testing and credential access techniques."
    )

@tool
def exploitation_worker(target: str) -> str:
    """Simulate an exploitation attempt on the target and return high-level outcome."""
    return f"[Exploitation] Successfully gained initial access to {target}."

@tool
def post_exploitation_worker() -> str:
    """Simulate post-exploitation activities such as persistence and lateral movement."""
    return "[Post-Exploitation] Established persistence and moved laterally."

@tool
def reporting_worker() -> str:
    """Generate a summary report of the red team engagement."""
    return "[Reporting] Final engagement report generated."

## 6. LLM and Specialized Agents

We use `gpt-4o-mini` for cost efficiency during development. Each agent receives only the single tool it is responsible for, enforcing separation of concerns.

In [4]:
# Create LLM and Specialized Agents

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=600
)

# Each agent only gets the tool it is responsible for
recon_agent        = create_agent(llm, [recon_worker])
exploit_agent      = create_agent(llm, [exploitation_worker])
post_exploit_agent = create_agent(llm, [post_exploitation_worker])
report_agent       = create_agent(llm, [reporting_worker])

print("Agents created successfully.")

Agents created successfully.


## 7. Running the Red Team Engagement with Human Oversight

The following cell executes the full four-phase engagement. Human approval is required after Phase 1 (Reconnaissance) and after Phase 2 (Exploitation). The exploitation and post-exploitation agents are explicitly instructed to use modern MITRE ATT&CK sub-technique IDs with proper hyperlink formatting.


Below is a representative output from a successful run of the simulator (target: WEB-PROD-07). The exploitation and post-exploitation phases correctly used modern MITRE ATT&CK sub-technique IDs with hyperlinks.

In [8]:
# Run the Red Team Engagement with Human-in-the-Loop

target = "WEB-PROD-07"

print(f"\n=== Starting Red Team Engagement on {target} (with Human Oversight) ===\n")

def get_content(msg):
    content = msg.content
    if isinstance(content, dict) and 'text' in content:
        return content['text']
    if isinstance(content, list):
        return "".join([b.get('text', str(b)) if isinstance(b, dict) else str(b) for b in content])
    return str(content)

# ============================================
# Phase 1: Reconnaissance
# ============================================
print(">>> Phase 1: Reconnaissance")
result = recon_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you."),
        HumanMessage(content=f"Perform reconnaissance on {target}")
    ]
})
recon_output = get_content(result["messages"][-1])
print(recon_output + "\n")

approval = input("Proceed to Exploitation Simulation phase? (yes/no): ").strip().lower()
if approval != "yes":
    print("\nEngagement stopped by human operator after reconnaissance.")
    exit()


# ============================================
# Phase 2: Exploitation Simulation
# ============================================
print("\n>>> Phase 2: Exploitation Simulation")
result = exploit_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you. Keep the response concise (3-4 sentences)."),
        HumanMessage(content=f"""Simulate an exploitation attempt on {target} in a controlled red team engagement.
Select 2-3 realistic techniques from MITRE ATT&CK that are relevant to a web server.

You **must** format each technique using this exact structure with no variations:
**Technique Name** [TXXXX.XXX](https://attack.mitre.org/techniques/TXXXX/XXX/) - short description.

Example:
**Phishing** [T1566.001](https://attack.mitre.org/techniques/T1566/001/) - description here.

Do not use empty brackets or put the ID outside the link. Keep the response concise and professional.""")
    ]
})
exploit_output = get_content(result["messages"][-1])
print(exploit_output + "\n")

approval = input("Proceed to Post-Exploitation Simulation phase? (yes/no): ").strip().lower()
if approval != "yes":
    print("\nEngagement stopped by human operator after exploitation simulation.")
    exit()


# ============================================
# Phase 3: Post-Exploitation Simulation
# ============================================
print("\n>>> Phase 3: Post-Exploitation Simulation")
result = post_exploit_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you. Keep the response concise (3-4 sentences)."),
        HumanMessage(content="""Briefly describe 2-3 key post-exploitation techniques used in red team engagements.
Focus on Persistence, Credential Access, and Lateral Movement.
For each technique, use this exact format:
**Technique Name** [TXXXX.XXX](https://attack.mitre.org/techniques/TXXXX/XXX/) - short description.
Use modern MITRE ATT&CK sub-technique IDs. Keep the response concise and professional.""")
    ]
})
post_output = get_content(result["messages"][-1])
print(post_output + "\n")

# ============================================
# Phase 4: Reporting
# ============================================
print(">>> Phase 4: Reporting")
report_prompt = f"""Generate a concise, professional red team engagement summary based ONLY on the following results:

Reconnaissance: {recon_output}

Exploitation: {exploit_output}

Post-Exploitation: {post_output}

Use exactly these section headings:
- Phases Completed
- Key Findings
- Limitations Encountered
- Overall Assessment

In the Key Findings section, list the techniques with their full MITRE ATT&CK IDs and hyperlinks.
In the Overall Assessment section, provide 1-2 specific, actionable recommendations based on the findings."""

result = report_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you."),
        HumanMessage(content=report_prompt)
    ]
})
print(get_content(result["messages"][-1]) + "\n")

print("=== Red Team Engagement Completed ===")


=== Starting Red Team Engagement on WEB-PROD-07 (with Human Oversight) ===

>>> Phase 1: Reconnaissance
Reconnaissance on WEB-PROD-07 has been completed with the following findings:

- **Open Ports:**
  - 22 (SSH)
  - 80 (HTTP)
  - 443 (HTTPS)
  - 3389 (RDP)

- **Services Detected:**
  - Apache 2.4.41
  - OpenSSH 8.2p1
  - Microsoft IIS 10.0

- **Potential Vulnerabilities:**
  - Outdated Apache version
  - Weak SSL/TLS configuration
  - Possible RDP exposure to the internet

- **Recommended Focus Areas:**
  - Web application testing
  - Credential access techniques

Proceed to Exploitation Simulation phase? (yes/no): yes

>>> Phase 2: Exploitation Simulation
**SQL Injection** [T1190](https://attack.mitre.org/techniques/T1190/) - An attacker exploits a vulnerability in a web application's database layer by injecting malicious SQL code.  
**Cross-Site Scripting (XSS)** [T1059.007](https://attack.mitre.org/techniques/T1059/007/) - An attacker injects malicious scripts into content from o

## 8. Key Observations

- **Human gates are effective but basic**: The current `input()` implementation works well for interactive notebook sessions but would need to be replaced with a proper oversight agent or callback mechanism for production or automated runs.
- **MITRE formatting quality is high**: When given strict formatting instructions, `gpt-4o-mini` reliably produces correctly linked sub-technique IDs. This is valuable for downstream parsing and SIEM integration.
- **Tool narrowness improves predictability**: Giving each agent only one tool reduces hallucination of tool names and makes outputs more consistent.
- **Reporting phase benefits from synthesis**: The final report agent produces professional, structured output when provided with clean phase results. This pattern scales well to more complex engagements.
- **Current limitations**: Tools return hardcoded/simulated results. Real-world use would require replacing these with actual scanning or exploitation modules (or safe wrappers).

## 9. Defensive Recommendations & Integration with Tool-Use Attacks

This simulator directly complements the attack primitives developed in `tool_use_agentic_attacks.ipynb`:

- The **Tool Schema Poisoning** and **Parameter Injection** techniques from the tool-use notebook can be incorporated as attack options inside the Exploitation phase of this simulator.
- The **Workflow Hijacking** and **Multi-Turn Goal Drift** patterns can be modeled as extended post-exploitation chains.
- Future versions of this simulator can include a “Tool-Use Attack Injector” agent that attempts to poison tool descriptions or parameters mid-engagement.

**Recommended Controls to Test with This Simulator**

| Priority | Control                        | How to Test with Simulator                          |
|----------|--------------------------------|-----------------------------------------------------|
| Critical | Human-in-the-Loop for high-impact actions | Run engagements with and without approval gates    |
| High     | Tool allow-listing + schema validation     | Inject poisoned tool descriptions and observe      |
| High     | MITRE-mapped logging of agent actions      | Parse the structured technique output into SIEM    |
| Medium   | Prompt hardening with refusal examples     | Test whether strict system prompts reduce success  |

## 10. Conclusion & Next Steps

This notebook provides a clean, professional foundation for multi-agent red team engagement simulation with meaningful human oversight. It successfully enforces phased execution, high-quality MITRE ATT&CK mapping, and the production of usable engagement reports.

**Next Steps**

1. **Migrate orchestration to LangGraph StateGraph** for true stateful multi-agent workflows with persistent memory between phases.
2. **Integrate tool-use attack primitives** from the companion notebook so the simulator can test both traditional TTPs and novel agentic attacks (schema poisoning, parameter injection, goal drift).
3. **Add automated scoring / judge agent** that evaluates engagement success, stealth, and MITRE coverage.
4. **Replace simulated tools** with safe wrappers around real (or containerized) target environments for higher fidelity.
5. **Build visualization layer** (engagement timeline, technique heat map, approval decision graph).
6. **Formal evaluation campaign** against detection systems (UEBA, NDR, EDR) using the generated artifacts.

This work directly supports ongoing research into agentic AI for both offensive security automation and defensive detection engineering.

Thank you for reviewing this notebook.